# Task [20/04-EDA-16.2]: Phân tích Sales (ADF Stationarity Test)

**Mục tiêu:** Kiểm tra tính dừng (stationarity) của chuỗi thời gian doanh thu (Revenue) để chuẩn bị cho các mô hình dự báo như ARIMA.

**Các bước thực hiện:**
1. Kiểm định **Augmented Dickey-Fuller (ADF)** trên cột `Revenue`.
2. Kiểm tra P-value:
    * Nếu $P < 0.05$: Chuỗi có tính dừng (Stationary).
    * Nếu $P \ge 0.05$: Chuỗi không dừng (Non-stationary) -> Cần lấy sai phân (Differencing).
3. Vẽ đồ thị **ACF (Autocorrelation)** và **PACF (Partial Autocorrelation)** động bằng Plotly để xác định các tham số $p, d, q$ cho model ARIMA.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, acf, pacf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

## 1. Đọc và chuẩn bị dữ liệu

In [ ]:
df = pl.read_csv('../Data/sales.csv')

# Chuyển Date sang datetime và sắp xếp
df = df.with_columns(pl.col('Date').str.to_date('%Y-%m-%d')).sort('Date')

# Chuyển sang pandas để dùng với statsmodels
df_pd = df.to_pandas().set_index('Date')
revenue = df_pd['Revenue']

print(f"Tổng số quan sát: {len(revenue)}")
print(f"Từ {revenue.index.min().date()} đến {revenue.index.max().date()}")

## 2. Kiểm định Augmented Dickey-Fuller (ADF)

Phát biểu giả thuyết:
- **H0 (Null Hypothesis):** Chuỗi thời gian KHÔNG có tính dừng (có chứa unit root).
- **H1 (Alternative Hypothesis):** Chuỗi thời gian CÓ tính dừng.
- Tiêu chí: Nếu `P-value < 0.05`, ta bác bỏ H0 và kết luận chuỗi có tính dừng.

In [ ]:
print("=== KẾT QUẢ KIỂM ĐỊNH ADF ===")
adf_result = adfuller(revenue.dropna())

print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'P-value:       {adf_result[1]:.4f}')
print('Critical Values:')
for key, value in adf_result[4].items():
    print(f'   {key}: {value:.4f}')

print("\n--- KẾT LUẬN ---")
if adf_result[1] < 0.05:
    print("P-value < 0.05 => Bác bỏ H0. Chuỗi CÓ TÍNH DỪNG (Stationary).")
    print("Bạn có thể sử dụng tham số d = 0 cho mô hình ARIMA.")
else:
    print("P-value >= 0.05 => Không đủ cơ sở bác bỏ H0. Chuỗi KHÔNG DỪNG (Non-stationary).")
    print("Cần thực hiện lấy sai phân (Differencing) bậc 1 (d = 1) hoặc biến đổi log trước khi đưa vào ARIMA.")

## 3. Hàm hỗ trợ vẽ biểu đồ ACF & PACF động (bằng Plotly)

Do `statsmodels` mặc định trả về biểu đồ tĩnh bằng matplotlib, ta viết hàm tùy chỉnh để tính toán các giá trị ACF/PACF (cùng khoảng tin cậy 95%) và đưa lên biểu đồ động.

In [ ]:
def plot_acf_pacf_plotly(series, lags=50, title_prefix=""):
    series = series.dropna()
    
    # Tính toán ACF và khoảng tin cậy 95% (alpha=0.05)
    acf_vals, acf_confint = acf(series, nlags=lags, alpha=0.05)
    # Tính toán PACF và khoảng tin cậy 95%
    pacf_vals, pacf_confint = pacf(series, nlags=lags, alpha=0.05)
    
    # Căn chỉnh lại khoảng tin cậy để vẽ shade area (xoay quanh trục 0 thay vì giá trị)
    acf_lower = acf_confint[:, 0] - acf_vals
    acf_upper = acf_confint[:, 1] - acf_vals
    pacf_lower = pacf_confint[:, 0] - pacf_vals
    pacf_upper = pacf_confint[:, 1] - pacf_vals
    
    x = np.arange(len(acf_vals))
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"ACF - {title_prefix}", f"PACF - {title_prefix}"))
    
    # --- 1. Plot ACF ---
    # Vẽ các dải dọc (stem lines)
    for i in range(len(acf_vals)):
        fig.add_trace(go.Scatter(x=[x[i], x[i]], y=[0, acf_vals[i]], mode='lines', 
                                 line=dict(color='rgba(31, 119, 180, 0.8)', width=2), showlegend=False), row=1, col=1)
    # Vẽ điểm (markers)
    fig.add_trace(go.Scatter(x=x, y=acf_vals, mode='markers', 
                             marker=dict(color='rgba(31, 119, 180, 1)', size=6), 
                             name='ACF', showlegend=False), row=1, col=1)
    # Vẽ vùng Confidence Interval
    fig.add_trace(go.Scatter(x=np.concatenate([x, x[::-1]]), 
                             y=np.concatenate([acf_upper, acf_lower[::-1]]), 
                             fill='toself', fillcolor='rgba(31, 119, 180, 0.2)', 
                             line=dict(color='rgba(255,255,255,0)'), 
                             name='95% CI', showlegend=False), row=1, col=1)
                             
    # --- 2. Plot PACF ---
    # Vẽ các dải dọc (stem lines)
    for i in range(len(pacf_vals)):
        fig.add_trace(go.Scatter(x=[x[i], x[i]], y=[0, pacf_vals[i]], mode='lines', 
                                 line=dict(color='rgba(255, 127, 14, 0.8)', width=2), showlegend=False), row=1, col=2)
    # Vẽ điểm (markers)
    fig.add_trace(go.Scatter(x=x, y=pacf_vals, mode='markers', 
                             marker=dict(color='rgba(255, 127, 14, 1)', size=6), 
                             name='PACF', showlegend=False), row=1, col=2)
    # Vẽ vùng Confidence Interval
    fig.add_trace(go.Scatter(x=np.concatenate([x, x[::-1]]), 
                             y=np.concatenate([pacf_upper, pacf_lower[::-1]]), 
                             fill='toself', fillcolor='rgba(255, 127, 14, 0.2)', 
                             line=dict(color='rgba(255,255,255,0)'), 
                             name='95% CI', showlegend=False), row=1, col=2)
    
    # Thêm đường zero-line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)
    
    fig.update_layout(height=500, title_text=f"Phân tích ACF & PACF - {title_prefix}", 
                      hovermode="closest", plot_bgcolor='white')
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', title_text="Lags")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    
    fig.show()

## 4. Vẽ đồ thị ACF và PACF

### Chuỗi gốc (Original)

In [ ]:
plot_acf_pacf_plotly(revenue, lags=50, title_prefix="Doanh thu Gốc")

### Chuỗi đã lấy sai phân bậc 1 (1st Differencing)
Mặc dù kết quả ADF có thể cho thấy chuỗi đã dừng ở mức ý nghĩa 5%, việc khảo sát thêm sai phân bậc 1 rất hữu ích khi xây dựng các mô hình ARIMA phức tạp.

In [ ]:
# Lấy sai phân bậc 1
revenue_diff = revenue.diff().dropna()

plot_acf_pacf_plotly(revenue_diff, lags=50, title_prefix="Sai phân Bậc 1 (1st Diff)")